# Train a small U-Net on a Kaggle GPU

This notebook is for a first Kaggle GPU session. It trains a semantic filament mask and, on a date holdout, prints semantic Dice and panoptic quality.

Panoptic quality uses `filament.instances` on the predicted mask and the downscaled grayscale image. It does not write `submission.csv` and it does not submit to Kaggle.

Full-disk frames are 2048×2048. Training uses 512×512. **512 is untuned.** It is a starting size so a free GPU session can finish. The mask is resized with nearest neighbor, so it stays binary and thin barbs can disappear. The image uses area averaging, because 512 divides 2048.

The GitHub repository stays private. Do not make it public.

Run the cells from the top.


## 1. Notebook settings

Open **Settings** in the notebook editor.

1. **Accelerator: GPU T4.** If the menu also lists GPU T4 x2, pick the single T4. One T4 is enough, and the free weekly GPU quota is shared across GPUs.
2. **Internet: On.** A later cell installs one package. Kaggle already has PyTorch. Do not add PyTorch to this repo's `pyproject.toml`.
3. The free weekly GPU quota runs out. When the last code cell prints the weights path, open the **Output** panel and download `unet.pt` before the session ends. The session disk is wiped when the session stops. Do not commit the weights.


## 2. Where the data already is

Create the notebook from the competition so this folder is attached. Do not download the images again. Do not train on the test images.

Train images, 2048×2048 JPEG:

`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/train_images`

Train JSON:

`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json`

Left unused on purpose:

`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images`

All four categories (Left, Right, Unidentifiable, Ambiguous) are one filament class. The target is the union of every polygon on that image. `category_id` is not read.


## 3. Private repo on `PYTHONPATH`

On your machine, zip this repo **without** `data/`, `.env`, or `kaggle.json`. On Kaggle: **Datasets → New Dataset**, set visibility to **Private**, upload the zip. In this notebook, **Add Input** and attach that dataset.

The next cell copies `src` to `/kaggle/working/src` and puts that directory on `PYTHONPATH`. A local checkout is used as it is, so the same file can be opened off Kaggle.


In [ ]:
import os
import shutil
import sys
from pathlib import Path


def _local_src() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "train_data.py").is_file():
            return src
    return None


def _content_src() -> Path | None:
    """A repo unzipped at /content or one folder under it."""
    root = Path("/content")
    if not root.is_dir():
        return None
    for src in (root / "src", *sorted(root.glob("*/src"))):
        if (src / "filament" / "train_data.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/train_data.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "train_data.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> None:
    """Put this repo's src on sys.path.

    Local checkouts are used as they are. Colab also accepts a repo
    unzipped under /content. On Kaggle the GitHub repo stays private:
    copy src from the private dataset into /kaggle/working/src and put
    that directory on PYTHONPATH.
    """
    local = _local_src() or _content_src()
    if local is not None:
        _put_on_path(local)
        print(f"src={local}")
        return
    destination = Path("/kaggle/working/src")
    if (destination / "filament" / "train_data.py").is_file():
        _put_on_path(destination)
        print(f"src={destination}")
        return
    source = _private_dataset_src()
    if source is None:
        if Path("/kaggle/working").is_dir():
            raise FileNotFoundError(
                "Could not find src/filament/train_data.py in the working "
                "directory or in a private dataset under /kaggle/input "
                "(any depth outside competitions, for example "
                "/kaggle/input/datasets/ignacioaguirrecandia/"
                "week1-solar-filament-segmentation/"
                "Solar-Filament-Segmentation-main/src). "
                "Upload a ZIP of this private repo as a private Kaggle "
                "dataset, without data/, .env, or kaggle.json. Do not make "
                "the GitHub repository public."
            )
        raise FileNotFoundError(
            "Could not find src/filament/train_data.py in the working "
            "directory or one level under /content (looked for /content/src "
            "and /content/*/src). Unzip the private repo under /content and "
            "cd into that folder, or leave the unzipped folder directly "
            "under /content. Do not make the GitHub repository public."
        )
    shutil.copytree(source, destination, dirs_exist_ok=True)
    _put_on_path(destination)
    print(f"src={destination}")


_prepare_src()
PYTHON = sys.executable

## 4. Install the segmentation library

`segmentation-models-pytorch` version **0.6.0**, MIT license.

The encoder is `resnet18` with `encoder_weights="imagenet"`. smp 0.6.0 pins resnet18/imagenet to Hugging Face repo `smp-hub/resnet18.imagenet` at revision `3f2325ff978283d47aa6a1d6878ca20565622683`, file `model.safetensors` (Hub license field `other`, not BSD-3; the model card names `https://download.pytorch.org/models/resnet18-5c106cde.pth` as the original file, and byte-identity with that `.pth` was not checked [VERIFICAR]), and an unauthenticated Hub download is expected. Do not pass `ssl` or `swsl`.

If the install fails, stop. Do not replace it with a weight whose license is research-only. Kaggle already has PyTorch. This cell does not install PyTorch.


In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Do not switch to an encoder weight whose license is research-only. "
        "Kaggle already has PyTorch. This cell only installs "
        "segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 5. Paths, GPU, and the date holdout

On Kaggle the weights path is `/kaggle/working/unet.pt` and nowhere else.

The holdout is the most recent 20% of calendar dates from `filament.split.split_by_date`. `date_captured` wins. A blank date falls back to `file_name`. A whole date stays on one side. Pixels are not split.

| Setting | Value |
| --- | --- |
| Side | 512, untuned, down from 2048 |
| Mask resize | nearest |
| Image resize | area (512 divides 2048) |
| Loss | Dice + BCE |
| Epochs | 12, untuned |
| Batch size | 4, untuned, chosen to fit a T4 at 512 |
| Seed | 0 |
| Optimizer | Adam, learning rate 1e-4, untuned |
| Holdout scores | semantic Dice, then PQ from `filament.instances` |

H-alpha is one band. The gray image is repeated to 3 channels so the torchvision ImageNet convolution stays intact. Inputs are divided by 255 and normalized with the ImageNet mean and standard deviation.

The next cell stops if there is no GPU, or if the GPU has under 6 GiB. The Asus laptop (about 4 GB of VRAM) is not the training machine.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import torch

from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.train_data import (
    TRAIN_SIDE,
    downscale_pair,
    instance_pq,
    semantic_target_from_annotations,
    split_training_images,
)

# Colab has no /kaggle/working. The backup section below names these paths.
if Path("/kaggle/working").is_dir():
    IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
    ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
    WEIGHTS_PATH = Path("/kaggle/working/unet.pt")
else:
    IMAGE_DIR = Path("/content/filament-data/train/train_images")
    ANNOTATIONS_PATH = Path(
        "/content/filament-data/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    )
    WEIGHTS_PATH = Path("/content/unet.pt")

print(f"images={IMAGE_DIR}")
print(f"annotations={ANNOTATIONS_PATH}")
print(f"weights={WEIGHTS_PATH}")
print(f"side={TRAIN_SIDE} untuned")
print("test/test_images is not used")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. On Kaggle set Accelerator to GPU T4. "
        "On Colab select a GPU runtime. The Asus (~4 GB VRAM) is not the "
        "training machine."
    )
vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
if vram_gib < 6:
    raise RuntimeError(
        f"This GPU has {vram_gib:.1f} GiB. The Asus (~4 GB VRAM) is not the "
        "training machine. Use a Kaggle T4 or a Colab GPU runtime."
    )
print(f"device={torch.cuda.get_device_name(0)} vram_gib={vram_gib:.1f}")

## 6. Load, union, and downscale

This cell reads the train JSON, holds out dates, then reads each JPEG once. Polygons are rasterized at the JSON size. Image and mask are then downscaled together to 512. That pass is the slow part. Do not point it at `test/test_images`.


In [ ]:
from torch.utils.data import Dataset

if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "On Kaggle, add the competition data. On Colab, upload the train "
        "images and the train JSON to those paths, or edit the path cell. "
        "Do not train on test/test_images."
    )

loaded = load_train_annotations(ANNOTATIONS_PATH)
split = split_training_images(loaded.images)
print(f"train images={len(split.train)} holdout images={len(split.holdout)}")
print(f"oldest train file={split.train[0].file_name}")
print(f"oldest holdout file={split.holdout[0].file_name}")
print("Holdout: most recent 20% of calendar dates, via split_by_date.")

# torchvision ImageNet normalization. The resnet18 weights were trained with it.
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)


class FilamentDisks(Dataset):
    """Gray disks and the union mask, stored at the training side.

    Area averaging returns float32 still on the 0-255 scale of the JPEG.
    ``grays`` is that array rounded to uint8 for the solar-disk gate.
    """

    def __init__(self, records: tuple) -> None:
        self.images: list[np.ndarray] = []
        self.grays: list[np.ndarray] = []
        self.masks: list[np.ndarray] = []
        total = len(records)
        for index, record in enumerate(records):
            if index % 100 == 0:
                print(f"prepared {index} / {total}")
            path = IMAGE_DIR / record.file_name
            disk = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if disk is None:
                raise FileNotFoundError(path)
            if disk.shape != (record.height, record.width):
                raise ValueError(
                    f"{record.file_name} is {disk.shape[0]}x{disk.shape[1]}, "
                    f"JSON says {record.height}x{record.width}"
                )
            mask = semantic_target_from_annotations(
                record.annotations,
                record.height,
                record.width,
            )
            small_disk, small_mask = downscale_pair(disk, mask, side=TRAIN_SIDE)
            self.images.append(small_disk)
            self.grays.append(np.clip(np.rint(small_disk), 0, 255).astype(np.uint8))
            self.masks.append(small_mask)
        print(f"prepared {total} / {total}")

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int):
        gray = self.images[index].astype(np.float32) / 255.0
        rgb = np.stack((gray, gray, gray), axis=0)
        tensor = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        target = self.masks[index].astype(np.float32)[None, ...]
        return (
            torch.from_numpy(np.ascontiguousarray(tensor)),
            torch.from_numpy(np.ascontiguousarray(target)),
        )


train_set = FilamentDisks(split.train)
holdout_set = FilamentDisks(split.holdout)

## 7. Train, then print holdout Dice and PQ

Dice + BCE, seed 0, `EPOCHS = 12`, batch size 4. All of that is untuned. The first run used 3 epochs and finished inside one free T4 session. Mean loss was still falling (1.4370, 1.2121, 1.0906), so 12 is four times that run. Wall-clock per epoch was not recorded. Expected runtime: [VERIFICAR].

The cell writes the weights first. It then prints semantic Dice and panoptic quality on the date holdout. PQ calls `filament.train_data.instance_pq` on the 512 prediction. That pools the annotator views of one file. The printed number is the mean of those per-file scores, which is not the competition PQ. The area cut is that module's default: 400 at 2048 px and 25 at 512 px. The close size is the module default, 0 (no closing). Both come from the 2026-11-01 holdout sweep (`docs/RESULTS.md`). This notebook does not pass another cutoff or kernel, does not write `submission.csv`, and does not submit to Kaggle.

The checkpoint is written only to the weights path from the path cell (`/kaggle/working/unet.pt` on Kaggle). Download it before the session ends. Do not commit it.


In [ ]:
import random

from torch.utils.data import DataLoader

SEED = 0
EPOCHS = 12
BATCH_SIZE = 4
LEARNING_RATE = 1e-4


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class DiceBceLoss(torch.nn.Module):
    """Sum of binary Dice and BCE-with-logits. Untuned."""

    def __init__(self) -> None:
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
        self.bce = torch.nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return self.dice(logits, target) + self.bce(logits, target)


def holdout_semantic_dice(model, loader, device: torch.device) -> float:
    """Dice of the thresholded probability against the union mask."""
    model.eval()
    intersection = 0.0
    predicted = 0.0
    actual = 0.0
    with torch.no_grad():
        for batch, target in loader:
            batch = batch.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            binary = (torch.sigmoid(model(batch)) >= 0.5).to(dtype=target.dtype)
            intersection += float((binary * target).sum())
            predicted += float(binary.sum())
            actual += float(target.sum())
    return (2.0 * intersection) / (predicted + actual + 1e-7)


set_seed(SEED)
device = torch.device("cuda")
try:
    model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    )
except Exception as exc:
    raise RuntimeError(
        "Stopping. resnet18 ImageNet weights did not load. Use "
        "encoder_weights='imagenet' only. smp 0.6.0 loads "
        "model.safetensors from smp-hub/resnet18.imagenet. "
        "The Hub license field is other. The model card names "
        "https://download.pytorch.org/models/resnet18-5c106cde.pth "
        "as the original file. Byte-identity with that .pth was "
        "not checked [VERIFICAR]. "
        "Do not switch to ssl or swsl. "
        f"Original error: {exc}"
    ) from exc

model = model.to(device)
loss_fn = DiceBceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
)
holdout_loader = DataLoader(
    holdout_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    seen = 0
    for batch, target in train_loader:
        batch = batch.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(batch), target)
        loss.backward()
        optimizer.step()
        total += float(loss.detach()) * batch.shape[0]
        seen += int(batch.shape[0])
    mean_loss = total / max(seen, 1)
    print(f"epoch {epoch}/{EPOCHS} mean_loss={mean_loss:.4f}")

score = holdout_semantic_dice(model, holdout_loader, device)

WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "encoder_name": "resnet18",
        "encoder_weights": "imagenet",
        "smp_version": smp.__version__,
        "side": int(TRAIN_SIDE),
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "in_channels": 3,
    },
    WEIGHTS_PATH,
)
print(f"wrote {WEIGHTS_PATH}")
print("Download this file before the session ends. Do not commit it.")

pq_scores: list[float] = []
model.eval()
with torch.no_grad():
    holdout_total = len(split.holdout)
    for index, record in enumerate(split.holdout):
        if index % 50 == 0:
            print(f"pq {index} / {holdout_total}")
        tensor, _target = holdout_set[index]
        logits = model(tensor.unsqueeze(0).to(device))
        semantic = (torch.sigmoid(logits)[0, 0] >= 0.5).to(dtype=torch.uint8)
        pq_scores.append(
            instance_pq(semantic.cpu().numpy(), holdout_set.grays[index], record)
        )
mean_pq = sum(pq_scores) / len(pq_scores)
print(f"holdout semantic dice={score:.4f}")
print(f"holdout pq={mean_pq:.4f} images={len(pq_scores)}")
print(
    "PQ is instance_pq per holdout file, then the mean of those scores. "
    "That mean is not the competition PQ. The area cut is the module "
    "default (400 at 2048, 25 at 512) and close size 0, from the "
    "2026-11-01 holdout sweep. This notebook does not write a submission "
    "and does not submit to Kaggle."
)

## 8. Colab backup

Use a **GPU** runtime (Runtime → Change runtime type → GPU). Internet has to be on for the same `segmentation-models-pytorch==0.6.0` install.

Unzip this private repo under `/content`. Then either change directory into that folder before running the cells, or leave the unzipped folder as a direct child of `/content`. The src cell looks in the working directory and one level under `/content` for `src/filament/train_data.py`.

```python
%cd /content/your-unzipped-repo
```

The path cell above checks for `/kaggle/working`. On Colab that directory is missing, so it uses:

```python
IMAGE_DIR = Path("/content/filament-data/train/train_images")
ANNOTATIONS_PATH = Path(
    "/content/filament-data/train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)
WEIGHTS_PATH = Path("/content/unet.pt")
```

Upload the competition **train** images and the train JSON into those folders, or edit the three lines in the path cell. If they are missing, the error names `IMAGE_DIR` and `ANNOTATIONS_PATH`, the paths it actually checked. Do not point them at `test/test_images`.

The Asus laptop (about 4 GB of VRAM) is not the training machine. The path cell refuses a GPU under 6 GiB.
